> **Tested on:** Python 3.11 · openai>=1.78.0 · pydantic>=2.11.4 · May 2026  
> **Typical API cost for this notebook:** ~$0.05–0.30 (vision tokens ~$0.001–0.003 per image depending on size and detail level)

# Lab 8 — Vision & Multimodal Agents

**Goal:** Build agents that see — image understanding, document parsing, visual reasoning chains, and RAG pipelines that retrieve by image content.

Week 5 (AutoGen Lab 2) showed how to send a single `MultiModalMessage`. This lab goes much deeper:

| Part | Topic |
|------|-------|
| 1 | Vision API fundamentals — URL vs base64, detail levels, token cost |
| 2 | Structured visual extraction with Pydantic |
| 3 | Multi-image reasoning chains |
| 4 | Document / screenshot agent (OCR-style parsing) |
| 5 | Vision RAG — retrieve images by content, answer questions over an image corpus |
| 6 | Production patterns — cost control, graceful degradation, caching |

The final section connects to Lab 3 (deployment) and Lab 4 (observability) — vision adds new cost and latency dimensions worth tracking.

In [ ]:
import base64
import json
import os
import re
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv()
client = OpenAI()
VISION_MODEL = "gpt-4o"  # gpt-4o-mini also supports vision at lower cost
print("Ready.")

---
## Part 1: Vision API Fundamentals

OpenAI's vision API accepts images in two ways:
- **URL** — the model fetches the image directly; fast, no encoding overhead
- **Base64** — embed image bytes inline; required for local files

The `detail` parameter controls how the model tiles the image:
- `"low"` — always 85 tokens regardless of image size; fast, good for thumbnails/icons
- `"high"` — tiles in 512×512 crops (each 170 tokens) + 85 base; better for dense text, charts
- `"auto"` — `"low"` if image ≤ 512×512, else `"high"`

**Cost rule of thumb:** A 1024×1024 image at `"high"` detail ≈ 765 tokens. At gpt-4o pricing ($2.50/1M input tokens) that's ~$0.002 per image.

In [ ]:
# --- 1a: Image from URL ---
# Using a stable public image (OpenAI's own documentation uses this pattern)
SAMPLE_CHART_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "3/3f/Bimodal_geological_map.png/320px-Bimodal_geological_map.png"
)

response = client.chat.completions.create(
    model=VISION_MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": SAMPLE_CHART_URL, "detail": "low"}},
                {"type": "text", "text": "Describe this image in one sentence."},
            ],
        }
    ],
    max_tokens=100,
)
print("URL image response:")
print(response.choices[0].message.content)
print(f"Tokens used: {response.usage.prompt_tokens} prompt / {response.usage.completion_tokens} completion")

In [ ]:
def encode_image_to_base64(image_path: str) -> str:
    """Read a local image file and return base64-encoded string."""
    with open(image_path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode("utf-8")


def vision_query(image_source: str, question: str, detail: str = "auto") -> dict:
    """Send a vision query. image_source can be a URL or a local file path."""
    if image_source.startswith(("http://", "https://")):
        image_content = {"type": "image_url", "image_url": {"url": image_source, "detail": detail}}
    else:
        ext = Path(image_source).suffix.lstrip(".")
        mime = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png", "gif": "image/gif", "webp": "image/webp"}.get(ext, "image/png")
        b64 = encode_image_to_base64(image_source)
        image_content = {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}", "detail": detail}}

    response = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[{"role": "user", "content": [image_content, {"type": "text", "text": question}]}],
        max_tokens=500,
    )
    return {
        "answer": response.choices[0].message.content,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
    }


# Demo: compare token cost at different detail levels
print("=== Detail level comparison ===")
for detail in ["low", "high", "auto"]:
    result = vision_query(SAMPLE_CHART_URL, "What colours appear in this image?", detail=detail)
    print(f"detail={detail:4s}  prompt_tokens={result['prompt_tokens']:4d}  answer_excerpt={result['answer'][:60]}...")

---
## Part 2: Structured Visual Extraction with Pydantic

Free-text descriptions are hard to process downstream. Use `response_format` with Pydantic to extract structured data directly from images — the same pattern as Week 2's structured outputs.

Example use cases:
- Receipt → `{vendor, total, date, line_items[]}`
- Business card → `{name, email, phone, company}`
- Chart → `{chart_type, x_axis_label, y_axis_label, data_points[]}`

In [ ]:
class DataPoint(BaseModel):
    label: str
    approximate_value: float | None = None


class ChartExtraction(BaseModel):
    chart_type: Literal["bar", "line", "pie", "scatter", "map", "table", "other"]
    title: str | None = None
    x_axis_label: str | None = None
    y_axis_label: str | None = None
    data_points: list[DataPoint] = Field(default_factory=list)
    key_insight: str = Field(description="One sentence summarising the main takeaway")
    confidence: Literal["high", "medium", "low"] = Field(
        description="How confident are you in the extracted values?"
    )


def extract_chart_data(image_url: str) -> ChartExtraction:
    response = client.beta.chat.completions.parse(
        model=VISION_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": image_url, "detail": "high"}},
                    {
                        "type": "text",
                        "text": (
                            "Extract structured data from this chart. "
                            "For data_points, extract the most prominent values you can read. "
                            "Set confidence=low if the image is unclear or values are hard to read precisely."
                        ),
                    },
                ],
            }
        ],
        response_format=ChartExtraction,
        max_tokens=800,
    )
    return response.choices[0].message.parsed


# Test on a simple bar chart image
BAR_CHART_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "1/1a/24701-nature-beauty.jpg/320px-24701-nature-beauty.jpg"
)
# Using the map image we already verified loads
extracted = extract_chart_data(SAMPLE_CHART_URL)
print("Structured extraction result:")
print(json.dumps(extracted.model_dump(), indent=2))

---
## Part 3: Multi-Image Reasoning Chains

A single API call can receive multiple images. This enables:
- **Before/after comparison** — "What changed between these two screenshots?"
- **Image series reasoning** — "What trend do these quarterly charts show?"
- **Cross-document extraction** — "Do these two receipts come from the same vendor?"

OpenAI supports up to ~20 images per request, but costs multiply — budget accordingly.

In [ ]:
def multi_image_compare(image_urls: list[str], question: str) -> str:
    """Send multiple images in one request for comparative reasoning."""
    content = []
    for i, url in enumerate(image_urls, 1):
        content.append({"type": "text", "text": f"Image {i}:"})
        content.append({"type": "image_url", "image_url": {"url": url, "detail": "low"}})
    content.append({"type": "text", "text": question})

    response = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[{"role": "user", "content": content}],
        max_tokens=600,
    )
    return response.choices[0].message.content


# Two different map/diagram images for comparison
IMAGE_A = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Bimodal_geological_map.png/320px-Bimodal_geological_map.png"
IMAGE_B = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"

comparison = multi_image_compare(
    [IMAGE_A, IMAGE_B],
    "Compare these two images. What type of image is each one, and what are the key visual differences?",
)
print("Multi-image comparison:")
print(comparison)

In [ ]:
# Multi-turn vision conversation — agent asks follow-up questions about the same image
def vision_conversation(image_url: str, questions: list[str]) -> list[dict]:
    """Multi-turn conversation about a single image."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_url, "detail": "high"}},
                {"type": "text", "text": questions[0]},
            ],
        }
    ]

    history = []
    for i, question in enumerate(questions):
        if i > 0:
            messages.append({"role": "user", "content": question})

        response = client.chat.completions.create(
            model=VISION_MODEL, messages=messages, max_tokens=300
        )
        answer = response.choices[0].message.content
        messages.append({"role": "assistant", "content": answer})
        history.append({"q": question, "a": answer})

    return history


# Three-turn conversation about one image
# NOTE: image is only sent once (in the first message); subsequent turns reference the same context
conv = vision_conversation(
    SAMPLE_CHART_URL,
    [
        "What does this image depict?",
        "What colour is most dominant?",
        "Would this image be useful in a geography textbook? Why or why not?",
    ],
)
print("Multi-turn vision conversation:")
for turn in conv:
    print(f"Q: {turn['q']}")
    print(f"A: {turn['a']}")
    print()

---
## Part 4: Document & Screenshot Agent

Vision models are excellent OCR replacements — they can read text from screenshots, scan PDFs (as images), and extract structured data from forms.

This section builds a `DocumentAgent` that:
1. Accepts a screenshot/document image
2. Classifies the document type (invoice, receipt, form, chart, etc.)
3. Routes to the right extraction schema based on type
4. Returns clean structured data

In [ ]:
class InvoiceData(BaseModel):
    vendor_name: str | None = None
    invoice_number: str | None = None
    date: str | None = None
    total_amount: str | None = None
    line_items: list[str] = Field(default_factory=list)
    notes: str | None = None


class FormData(BaseModel):
    fields: dict[str, str] = Field(
        default_factory=dict,
        description="Field name → extracted value pairs from the form"
    )
    form_title: str | None = None


class DocumentClassification(BaseModel):
    doc_type: Literal["invoice", "receipt", "form", "chart", "screenshot", "photo", "other"]
    confidence: Literal["high", "medium", "low"]
    contains_text: bool
    language: str = Field(default="english", description="Primary language of any text")


def classify_document(image_url: str) -> DocumentClassification:
    response = client.beta.chat.completions.parse(
        model=VISION_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": image_url, "detail": "low"}},
                    {"type": "text", "text": "Classify this document image."},
                ],
            }
        ],
        response_format=DocumentClassification,
        max_tokens=200,
    )
    return response.choices[0].message.parsed


def extract_document(image_url: str) -> dict:
    """Classify then route to the right extraction schema."""
    classification = classify_document(image_url)
    print(f"  Classified as: {classification.doc_type} (confidence={classification.confidence})")

    if classification.doc_type in ("invoice", "receipt"):
        response = client.beta.chat.completions.parse(
            model=VISION_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": image_url, "detail": "high"}},
                        {"type": "text", "text": "Extract all invoice/receipt fields."},
                    ],
                }
            ],
            response_format=InvoiceData,
            max_tokens=600,
        )
        return {"type": classification.doc_type, "data": response.choices[0].message.parsed.model_dump()}

    elif classification.doc_type in ("chart",):
        return {"type": "chart", "data": extract_chart_data(image_url).model_dump()}

    else:
        # Generic text extraction for screenshots, forms, other
        response = client.chat.completions.create(
            model=VISION_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": image_url, "detail": "high"}},
                        {
                            "type": "text",
                            "text": "Extract all readable text from this image, preserving structure where possible.",
                        },
                    ],
                }
            ],
            max_tokens=800,
        )
        return {"type": classification.doc_type, "data": {"text": response.choices[0].message.content}}


# Test the document agent
print("Document agent test:")
result = extract_document(SAMPLE_CHART_URL)
print(json.dumps(result, indent=2))

---
## Part 5: Vision RAG — Retrieve Images by Content

Standard RAG retrieves text chunks by embedding similarity. Vision RAG does the same for images:

1. **Index** — for each image, generate a text description (via vision model) and embed that description
2. **Retrieve** — embed the user query, find the most similar image descriptions
3. **Answer** — send the retrieved images + query to the vision model

This lets you answer questions over an image corpus without storing raw images in a vector database — you store text descriptions and fetch images on demand.

```
Images ─→ Vision LLM ─→ Text Descriptions ─→ Embeddings ─→ Vector Store
                                                                  ↑
User query ─→ Embed ─→ Similarity search ─────────────────────────┘
                                              ↓
                         Fetch matched images + query ─→ Vision LLM ─→ Answer
```

In [ ]:
import numpy as np

EMBED_MODEL = "text-embedding-3-small"


def embed(text: str) -> list[float]:
    return client.embeddings.create(model=EMBED_MODEL, input=text).data[0].embedding


def cosine_similarity(a: list[float], b: list[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


class ImageIndexEntry:
    def __init__(self, image_id: str, image_url: str, description: str):
        self.image_id = image_id
        self.image_url = image_url
        self.description = description
        self.embedding = embed(description)


class VisionRAG:
    def __init__(self):
        self.index: list[ImageIndexEntry] = []

    def add_image(self, image_id: str, image_url: str):
        """Describe image with vision model and add to index."""
        print(f"  Indexing {image_id}...")
        response = client.chat.completions.create(
            model=VISION_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": image_url, "detail": "low"}},
                        {
                            "type": "text",
                            "text": (
                                "Describe this image in 2-3 sentences, "
                                "focusing on content, colors, and what someone might search for to find it. "
                                "Be specific and factual."
                            ),
                        },
                    ],
                }
            ],
            max_tokens=150,
        )
        description = response.choices[0].message.content
        entry = ImageIndexEntry(image_id, image_url, description)
        self.index.append(entry)
        print(f"    Description: {description[:80]}...")

    def retrieve(self, query: str, top_k: int = 2) -> list[ImageIndexEntry]:
        """Find the most relevant images for a text query."""
        query_embedding = embed(query)
        scored = [
            (cosine_similarity(query_embedding, entry.embedding), entry)
            for entry in self.index
        ]
        scored.sort(key=lambda x: x[0], reverse=True)
        return [entry for _, entry in scored[:top_k]]

    def answer(self, query: str, top_k: int = 2) -> str:
        """Retrieve relevant images and answer the query using them."""
        retrieved = self.retrieve(query, top_k)
        print(f"  Retrieved images: {[e.image_id for e in retrieved]}")

        content = [{"type": "text", "text": f"Answer the following question using the provided images: {query}\n"}]
        for i, entry in enumerate(retrieved, 1):
            content.append({"type": "text", "text": f"Image {i} (id={entry.image_id}):"})
            content.append({"type": "image_url", "image_url": {"url": entry.image_url, "detail": "low"}})

        response = client.chat.completions.create(
            model=VISION_MODEL,
            messages=[{"role": "user", "content": content}],
            max_tokens=400,
        )
        return response.choices[0].message.content


print("Building Vision RAG index (4 images)...")
rag = VisionRAG()

# Index a small corpus of diverse images
IMAGE_CORPUS = [
    ("geological-map", "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Bimodal_geological_map.png/320px-Bimodal_geological_map.png"),
    ("png-transparency", "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"),
    ("color-wheel", "https://upload.wikimedia.org/wikipedia/commons/thumb/5/5f/Color_circle_%28hue-sat%29.png/240px-Color_circle_%28hue-sat%29.png"),
    ("gradient-example", "https://upload.wikimedia.org/wikipedia/commons/thumb/0/07/Gradient_gardients.jpg/320px-Gradient_gardients.jpg"),
]

for img_id, img_url in IMAGE_CORPUS:
    rag.add_image(img_id, img_url)

In [ ]:
# Query the vision RAG system
queries = [
    "Which image shows a scientific or geographical diagram?",
    "Find an image related to color or visual spectrum",
]

for query in queries:
    print(f"\nQuery: {query}")
    answer = rag.answer(query, top_k=2)
    print(f"Answer: {answer}")

---
## Part 6: Production Patterns for Vision Agents

Vision adds three new production concerns:

1. **Cost control** — images are expensive; use `detail="low"` for thumbnails, cache descriptions
2. **Graceful degradation** — if vision model is unavailable or image fails to load, fall back to text-only
3. **Content moderation** — images can contain harmful content; check before processing

In [ ]:
import functools
import time


# --- 6a: Image description cache ---
# Descriptions are deterministic for a given image URL + detail level.
# Cache them so repeated queries over the same corpus don't re-call the vision API.
_description_cache: dict[str, str] = {}


def describe_image_cached(image_url: str, detail: str = "low") -> str:
    cache_key = f"{image_url}::{detail}"
    if cache_key in _description_cache:
        return _description_cache[cache_key]

    response = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": image_url, "detail": detail}},
                    {"type": "text", "text": "Describe this image concisely."},
                ],
            }
        ],
        max_tokens=200,
    )
    description = response.choices[0].message.content
    _description_cache[cache_key] = description
    return description


# Demo: second call is free
t0 = time.perf_counter()
desc1 = describe_image_cached(SAMPLE_CHART_URL)
t1 = time.perf_counter()
desc2 = describe_image_cached(SAMPLE_CHART_URL)  # cache hit
t2 = time.perf_counter()

print(f"First call:  {t1 - t0:.2f}s  (API call)")
print(f"Second call: {t2 - t1:.4f}s (cache hit)")
print(f"Cache size: {len(_description_cache)} entry")

In [ ]:
# --- 6b: Graceful degradation ---
# If a URL returns an error or the vision model fails, fall back to a text-only response.

def safe_vision_query(image_url: str, question: str) -> dict:
    """Vision query with fallback to text-only if image loading fails."""
    try:
        result = vision_query(image_url, question, detail="low")
        return {"source": "vision", **result}
    except Exception as e:
        # Image failed — answer from text alone with a caveat
        print(f"  Vision failed ({type(e).__name__}: {e}), falling back to text-only")
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # cheaper model for fallback
            messages=[
                {
                    "role": "user",
                    "content": (
                        f"I tried to show you an image at {image_url} but it failed to load. "
                        f"Please answer this question as best you can without seeing it: {question}"
                    ),
                }
            ],
            max_tokens=200,
        )
        return {
            "source": "text_fallback",
            "answer": response.choices[0].message.content,
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
        }


# Test with a broken URL
print("Testing graceful degradation with broken URL:")
result = safe_vision_query("https://example.com/nonexistent_image_12345.png", "What colours are in this chart?")
print(f"Source: {result['source']}")
print(f"Answer: {result['answer'][:120]}...")

In [ ]:
# --- 6c: Token cost estimator for vision requests ---
# Use this before sending a batch to estimate cost and flag expensive requests.

def estimate_vision_tokens(width: int, height: int, detail: str = "auto") -> int:
    """Estimate token cost for a vision image based on OpenAI's tiling formula."""
    if detail == "low":
        return 85

    # For high/auto: scale so longest side <= 2048, shortest side <= 768
    if max(width, height) > 2048:
        scale = 2048 / max(width, height)
        width, height = int(width * scale), int(height * scale)

    if min(width, height) > 768:
        scale = 768 / min(width, height)
        width, height = int(width * scale), int(height * scale)

    # Count 512×512 tiles
    tiles_w = -(-width // 512)   # ceiling division
    tiles_h = -(-height // 512)
    total_tiles = tiles_w * tiles_h

    return 85 + 170 * total_tiles


# Cost table for common image sizes
GPT4O_INPUT_COST_PER_TOKEN = 2.50 / 1_000_000  # $2.50 per 1M tokens

print("Vision token cost estimates (gpt-4o):")
print(f"{'Size':20s} {'Detail':6s} {'Tokens':8s} {'Cost':10s}")
print("-" * 50)
test_cases = [
    (256, 256, "low"),
    (256, 256, "high"),
    (512, 512, "high"),
    (1024, 768, "high"),
    (1920, 1080, "high"),
    (4096, 4096, "high"),
]
for w, h, detail in test_cases:
    tokens = estimate_vision_tokens(w, h, detail)
    cost = tokens * GPT4O_INPUT_COST_PER_TOKEN
    print(f"{w}x{h:20d}  {detail:6s} {tokens:8d} ${cost:.5f}")

---
## Part 7: Full Multimodal Agent

Combining everything: a `MultimodalAgent` that can handle user messages containing text, URLs, or local file paths, routing each content type appropriately.

This is the pattern you'd use in a production assistant that handles mixed inputs — the agent decides whether to invoke vision, text-only, or tools based on what the user sends.

In [ ]:
class MultimodalAgent:
    """Agent that handles mixed text/image inputs and maintains conversation history."""

    def __init__(self, system_prompt: str = "You are a helpful assistant that can analyze images and answer questions."):
        self.system_prompt = system_prompt
        self.history: list[dict] = []
        self.total_tokens = 0

    def _parse_content(self, user_input: str, images: list[str] | None = None) -> list[dict] | str:
        """Build content list, attaching any images."""
        if not images:
            return user_input  # plain string — no vision cost

        content = []
        for img in images:
            if img.startswith(("http://", "https://")):
                content.append({"type": "image_url", "image_url": {"url": img, "detail": "auto"}})
            elif Path(img).exists():
                ext = Path(img).suffix.lstrip(".")
                mime = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png"}.get(ext, "image/png")
                b64 = encode_image_to_base64(img)
                content.append({"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}", "detail": "auto"}})
            else:
                print(f"  Warning: image not found, skipping: {img}")

        content.append({"type": "text", "text": user_input})
        return content

    def chat(self, user_input: str, images: list[str] | None = None) -> str:
        content = self._parse_content(user_input, images)
        self.history.append({"role": "user", "content": content})

        response = client.chat.completions.create(
            model=VISION_MODEL,
            messages=[{"role": "system", "content": self.system_prompt}] + self.history,
            max_tokens=600,
        )

        answer = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": answer})
        self.total_tokens += response.usage.total_tokens
        return answer

    def reset(self):
        self.history = []
        self.total_tokens = 0


# Demo: multi-turn agent that handles both text and image turns
agent = MultimodalAgent(
    system_prompt="You are a visual analysis assistant. When given images, describe and analyse them precisely. Be concise."
)

print("Turn 1 — Text only")
r1 = agent.chat("What's the difference between a geological map and a topographic map?")
print(f"Agent: {r1}\n")

print("Turn 2 — With image")
r2 = agent.chat("Here's an example. How does it compare to your description?", images=[SAMPLE_CHART_URL])
print(f"Agent: {r2}\n")

print("Turn 3 — Follow-up (no image needed — already in context)")
r3 = agent.chat("What colours are used to represent different geological features in that map?")
print(f"Agent: {r3}\n")

print(f"Total tokens used across 3 turns: {agent.total_tokens}")

---
## Summary

### What we built

| Component | What it does |
|-----------|-------------|
| `vision_query()` | One-shot image question with URL/base64/detail control |
| `extract_chart_data()` | Structured Pydantic extraction from charts |
| `multi_image_compare()` | Comparative reasoning across multiple images |
| `extract_document()` | Classify-then-extract document routing agent |
| `VisionRAG` | Full RAG pipeline: index images by description, retrieve by query |
| `describe_image_cached()` | Caching layer — no repeat API calls for the same image |
| `safe_vision_query()` | Graceful degradation to text-only on vision failure |
| `estimate_vision_tokens()` | Cost estimator for capacity planning |
| `MultimodalAgent` | Full agent with mixed text/image history |

### When to use which

| Scenario | Pattern |
|----------|---------|
| One-off image question | `vision_query()` |
| Extract structured data | `client.beta.chat.completions.parse()` with Pydantic schema |
| Ongoing conversation about images | `MultimodalAgent` |
| Q&A over an image corpus | `VisionRAG` |
| Unknown document type | `extract_document()` (classify-then-route) |

### Connecting to other labs

- **Lab 2 (Evals):** Add a `vision_faithfulness` metric — does the answer only claim things visible in the image?
- **Lab 3 (Production):** Vision requests are 5–20× more expensive than text; add per-session vision token budgets alongside the text token budget.
- **Lab 4 (Observability):** Log `vision_tokens_used` separately — it's the dominant cost driver in multimodal agents.
- **Lab 5 (Security):** Prompt injection can arrive via text embedded in images ("ignore previous instructions"). The two-stage filter from Lab 5 should also describe image content before classification.